In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sachinsingh152/mine11/bert_ready_data.csv


In [2]:
# ============================================
# ⚡ FAST SENTIMENT MODEL (DISTILBERT)
# ============================================

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load data
df = pd.read_csv("/kaggle/input/datasets/sachinsingh152/mine11/bert_ready_data.csv")

# Fix columns if needed
TEXT_COL = "text"
LABEL_COL = "label"

# Convert labels
if df[LABEL_COL].dtype == "object":
    df[LABEL_COL] = df[LABEL_COL].map({'negative': 0, 'positive': 1})

train_df, test_df = train_test_split(df, test_size=0.2)

# Tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class DatasetClass(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            max_length=64,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx])
        }

train_loader = DataLoader(DatasetClass(train_df[TEXT_COL], train_df[LABEL_COL]), batch_size=16, shuffle=True)
test_loader = DataLoader(DatasetClass(test_df[TEXT_COL], test_df[LABEL_COL]), batch_size=16)

# ============================================
# ⚡ MODEL (FREEZE BERT = SUPER FAST)
# ============================================

class FastModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        
        # 🔥 Freeze BERT (KEY SPEED BOOST)
        for param in self.bert.parameters():
            param.requires_grad = False

        self.fc = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = outputs.last_hidden_state[:, 0]   # CLS token
        return self.fc(x)

model = FastModel().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# ============================================
# ⚡ TRAIN (VERY FAST NOW)
# ============================================

EPOCHS = 3  # reduce epochs

for epoch in range(EPOCHS):
    model.train()
    loop = tqdm(train_loader)

    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, mask)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        loop.set_postfix(loss=loss.item())

# ============================================
# ⚡ EVAL
# ============================================

model.eval()
preds, y_true = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)

        outputs = model(input_ids, mask)
        preds.extend(torch.argmax(outputs, 1).cpu().numpy())
        y_true.extend(batch['label'].numpy())

print("Accuracy:", accuracy_score(y_true, preds))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 38013/38013 [21:47<00:00, 29.07it/s, loss=0.557]


Accuracy: 0.6040999401516597


In [3]:
import os

# Create a directory to store the model
SAVE_DIR = "/kaggle/working/distilbert_sentiment"
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# 1. Save the DistilBERT part (Architecture + Config)
model.bert.save_pretrained(SAVE_DIR)

# 2. Save the Tokenizer (Essential for preprocessing new text)
tokenizer.save_pretrained(SAVE_DIR)

# 3. Save the Classification Head weights (The Linear layer you trained)
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "full_model_state.pth"))

print(f"\n✅ All files saved to {SAVE_DIR}")

# Zip the folder so it's a single file for easy download
import shutil
shutil.make_archive("my_distilbert_model", 'zip', SAVE_DIR)
print("📦 Model zipped: my_distilbert_model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ All files saved to /kaggle/working/distilbert_sentiment
📦 Model zipped: my_distilbert_model.zip
